# STFT


In [1]:
import numpy as np
from scipy.io import wavfile
from scipy.signal.windows import hann
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt
import os
import hashlib

In [2]:
def window_nonzero(window_function, segment_length):
    zero_exist = 1
    zero_count = 0
    
    window_vector = window_function(segment_length + zero_count)

    while zero_exist:
        start = int(zero_count / 2)
        stop = int(len(window_vector) - zero_count / 2)
        window_vector = window_vector[start:stop]

        zero_count = len(window_vector) - np.count_nonzero(window_vector)

        if zero_count > 0:
            window_vector = window_function(segment_length + zero_count)
        else:
            zero_exist = 0

    return window_vector


def create_overlapping_segments(x, segment_length, shift_length):
    if type(x) is not np.ndarray:
        raise ValueError("x is not numpy array")
    if segment_length > x.shape[0]:
        raise ValueError("segment_length is greater than x.shape[0]")
    if shift_length <= 0:
        raise ValueError("shift_length <= 0")
    if shift_length > segment_length:
        raise ValueError("shift_length > segment_length")

    x = np.squeeze(x)

    start_list = np.arange(0, x.shape[0], shift_length)
    stop_list = start_list + segment_length

    index = [i <= x.shape[0] for i in stop_list]
    start_list = start_list[index]
    stop_list = stop_list[index]

    if stop_list[-1] != x.shape[0]:
        stop_list = np.append(stop_list, x.shape[0])
        start_list = np.append(start_list, x.shape[0] - segment_length)

    x_segments = [x[start:stop, ...] for start, stop in zip(start_list, stop_list)]
    x_segments = np.stack(x_segments, axis=1)

    return x_segments, start_list, stop_list

#### прямое stft 

In [3]:
def stft(x, segment_length, segment_length_padded, shift_length, window_function):
    if type(x) is not np.ndarray:
        raise ValueError("x is not numpy array")
    if segment_length > x.shape[0]:
        raise ValueError("segment_length is greater than x.shape[0]")
    if shift_length <= 0:
        raise ValueError("shift_length <= 0")
    if shift_length > segment_length:
        raise ValueError("shift_length > segment_length")
    if segment_length_padded < segment_length:
        raise ValueError("segment_length_padded < segment_length")

    window_vector = window_nonzero(window_function, segment_length)

    x_segments, start_list, stop_list = create_overlapping_segments(
        x, segment_length, shift_length
    )

    window_array = np.ones(x_segments.shape)
    for i in range(window_array.shape[0]):
        window_array[i] = window_array[i] * window_vector[i]

    x_segments = window_array * x_segments

    x_stft = np.fft.rfft(x_segments, n=segment_length_padded, axis=0)

    return x_stft, start_list, stop_list

#### обратное stft

In [4]:
def istft(x_stft, segment_length, segment_length_padded,
          start_list, stop_list, original_size,
          window_function, p):

    x_segments = np.fft.irfft(x_stft, n=segment_length_padded, axis=0)
    x_segments = x_segments[0:segment_length]

    window_vector = window_nonzero(window_function, segment_length)

    window_array = np.ones(x_segments.shape)
    for i in range(window_array.shape[0]):
        window_array[i] = window_array[i] * window_vector[i]
    window_array = window_array ** (p - 1)

    x_segments = window_array * x_segments

    window_overlap_add = np.zeros(original_size[0])
    number_segments = len(start_list)
    for i in range(number_segments):
        window_overlap_add[start_list[i]:stop_list[i]] += window_vector ** p
    window_overlap_add = (window_overlap_add) ** -1

    x = np.zeros(original_size)
    for i, (start, stop) in enumerate(zip(start_list, stop_list)):
        x[start:stop, ...] += x_segments[:, i, ...]

    window_overlap_add_array = np.zeros(original_size)
    for i in range(x.shape[0]):
        window_overlap_add_array[i] = window_overlap_add[i]
    x = x * window_overlap_add_array

    return x

sign flip

In [5]:
KEY_PATH = "./key/key.bin"

def load_key_bytes(path):
    with open(path, "rb") as f:
        return f.read()

def _seed_from_key(key_bytes, salt):
    h = hashlib.sha256()
    h.update(key_bytes)
    h.update(str(salt).encode("utf-8"))
    return int.from_bytes(h.digest()[:8], "big", signed=False)

def sign_mask_vector(size, key_bytes, salt=""):
    rng = np.random.default_rng(_seed_from_key(key_bytes, salt))
    return rng.choice(np.array([-1.0, 1.0]), size=size).astype(np.float64)

def sign_mask_blocks(n_samples, block_size, key_bytes, salt=""):
    rng = np.random.default_rng(_seed_from_key(key_bytes, salt))
    n_blocks = (n_samples + block_size - 1) // block_size
    signs = rng.choice(np.array([-1.0, 1.0]), size=n_blocks).astype(np.float64)
    return np.repeat(signs, block_size)[:n_samples]

key_bytes = load_key_bytes(KEY_PATH)

In [6]:
def stft_signflip(X, key_bytes):
    X2 = X.copy()

    if X2.shape[0] > 2:
        mask = sign_mask_vector(X2.shape[0] - 2, key_bytes, salt=f"stft_sign_{X2.shape[0]}")
        X2[1:-1, :] = X2[1:-1, :] * mask[:, None]

    return X2

### main

In [7]:
def load_perm_key(path, size):
    perm = np.fromfile(path, dtype=np.int64)

    if len(perm) != size:
        raise ValueError("bad key length")

    if np.unique(perm).shape[0] != size:
        raise ValueError("key is not a valid permutation")

    if perm.min() != 0 or perm.max() != size - 1:
        raise ValueError("key values out of range")

    return perm

def write_wav(path, fs, sig):
    sig = sig.astype(np.float64)
    m = np.max(np.abs(sig)) + 1e-12
    sig = sig / m
    wavfile.write(path, fs, (sig * 32767).astype(np.int16))

In [8]:
segment_length = 1024
segment_length_padded = 1024
shift_length = 512
window_function = hann
p = 1

DATASET_DIR = "../../dataset/test_sounds"

OUT_ENC_INVERT_DIR = "./assets/encrypted/invert"
OUT_DEC_INVERT_DIR = "./assets/decrypted/invert"

OUT_ENC_SHUFFLE_DIR = "./assets/encrypted/shuffle"
OUT_DEC_SHUFFLE_DIR = "./assets/decrypted/shuffle"

OUT_ENC_SIGNFLIP_DIR = "./assets/encrypted/signflip"
OUT_DEC_SIGNFLIP_DIR = "./assets/decrypted/signflip"

os.makedirs(OUT_ENC_INVERT_DIR, exist_ok=True)
os.makedirs(OUT_DEC_INVERT_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SHUFFLE_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SHUFFLE_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SIGNFLIP_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SIGNFLIP_DIR, exist_ok=True)

N_FILES = 23

for i in range(1, N_FILES + 1):
    in_path = os.path.join(DATASET_DIR, f"test_sound{i:1d}.wav")

    fs, data = wavfile.read(in_path)

    if data.ndim > 1:
        data = data[:, 0]

    data = data.astype(np.float64)
    original_size = data.shape

    X_stft, start_list, stop_list = stft(
        data,
        segment_length,
        segment_length_padded,
        shift_length,
        window_function
    )

    X_inv = X_stft[::-1, :]

    inv_audio = istft(
        X_inv,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    X_inv_dec = X_inv[::-1, :]

    inv_dec = istft(
        X_inv_dec,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    write_wav(os.path.join(OUT_ENC_INVERT_DIR, f"test_sound{i:1d}_invert_enc.wav"), fs, inv_audio)
    write_wav(os.path.join(OUT_DEC_INVERT_DIR, f"test_sound{i:1d}_invert_dec.wav"), fs, inv_dec)

    perm = np.random.permutation(X_stft.shape[0])
    inv_perm = np.argsort(perm)

    X_shuffle = X_stft[perm, :]

    shuffle_audio = istft(
        X_shuffle,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    X_shuffle_dec = X_shuffle[inv_perm, :]

    shuffle_dec = istft(
        X_shuffle_dec,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    write_wav(os.path.join(OUT_ENC_SHUFFLE_DIR, f"test_sound{i:1d}_shuffle_enc.wav"), fs, shuffle_audio)
    write_wav(os.path.join(OUT_DEC_SHUFFLE_DIR, f"test_sound{i:1d}_shuffle_dec.wav"), fs, shuffle_dec)

    X_sign = stft_signflip(X_stft, key_bytes)

    sign_audio = istft(
        X_sign,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    X_sign_dec = stft_signflip(X_sign, key_bytes)

    sign_dec = istft(
        X_sign_dec,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    write_wav(os.path.join(OUT_ENC_SIGNFLIP_DIR, f"test_sound{i:1d}_signflip_enc.wav"), fs, sign_audio)
    write_wav(os.path.join(OUT_DEC_SIGNFLIP_DIR, f"test_sound{i:1d}_signflip_dec.wav"), fs, sign_dec)

In [ ]:
segment_length = 1024
segment_length_padded = 1024
shift_length = 512  # 50% overlap
window_function = hann
p = 1  # overlap-add

In [ ]:
fs, data = wavfile.read("../dataset/test_sounds/pets.wav")

if data.ndim > 1:
    data = data[:, 0]

data = data.astype(np.float64)
original_size = data.shape

### просто прямое и обратное

In [ ]:
X_stft, start_list, stop_list = stft(
    data,
    segment_length,
    segment_length_padded,
    shift_length,
    window_function
)

reconstructed = istft(
    X_stft,
    segment_length,
    segment_length_padded,
    start_list,
    stop_list,
    original_size,
    window_function,
    p
)

wavfile.write("./assets/stft_clean.wav", fs, (reconstructed / np.max(np.abs(reconstructed)) * 32767).astype(np.int16))

#### инверсия частот

In [ ]:
X_inv = X_stft[::-1, :]

inv_audio = istft(
    X_inv,
    segment_length,
    segment_length_padded,
    start_list,
    stop_list,
    original_size,
    window_function,
    p
)

wavfile.write("./assets/enc/stft_inverted.wav", fs, (inv_audio / np.max(np.abs(inv_audio)) * 32767).astype(np.int16))

# дешифрование - повторная инверсия
X_inv_dec = X_inv[::-1, :]

inv_dec = istft(
    X_inv_dec,
    segment_length,
    segment_length_padded,
    start_list,
    stop_list,
    original_size,
    window_function,
    p
)

wavfile.write("./assets/dec/stft_inverted_dec.wav", fs, (inv_dec / np.max(np.abs(inv_dec)) * 32767).astype(np.int16))

### Полный shuffle частот

In [ ]:
perm = np.random.permutation(X_stft.shape[0])
X_shuffle = X_stft[perm, :]

shuffle_audio = istft(
    X_shuffle,
    segment_length,
    segment_length_padded,
    start_list,
    stop_list,
    original_size,
    window_function,
    p
)

wavfile.write("./assets/enc/stft_shuffle.wav", fs, (shuffle_audio / np.max(np.abs(shuffle_audio)) * 32767).astype(np.int16))

# дешифрование
inv_perm = np.argsort(perm)
X_shuffle_dec = X_shuffle[inv_perm, :]

shuffle_dec = istft(
    X_shuffle_dec,
    segment_length,
    segment_length_padded,
    start_list,
    stop_list,
    original_size,
    window_function,
    p
    )

wavfile.write("./assets/dec/stft_shuffle_dec.wav", fs, (shuffle_dec / np.max(np.abs(shuffle_dec)) * 32767).astype(np.int16))

### блочная перестановка

In [24]:
segment_length = 1024
segment_length_padded = 1024
shift_length = 512
window_function = hann
p = 1

DATASET_DIR = "../../dataset/test_sounds"

OUT_ENC_INVERT_DIR = "./assets/encrypted/invert"
OUT_DEC_INVERT_DIR = "./assets/decrypted/invert"

OUT_ENC_SHUFFLE_DIR = "./assets/encrypted/shuffle"
OUT_DEC_SHUFFLE_DIR = "./assets/decrypted/shuffle"

os.makedirs(OUT_ENC_INVERT_DIR, exist_ok=True)
os.makedirs(OUT_DEC_INVERT_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SHUFFLE_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SHUFFLE_DIR, exist_ok=True)

N_FILES = 23

for i in range(1, N_FILES + 1):

    in_path = os.path.join(DATASET_DIR, f"test_sound{i}.wav")

    fs, data = wavfile.read(in_path)

    if data.ndim > 1:
        data = data[:,0]

    data = data.astype(np.float64)
    original_size = data.shape

    X_stft, start_list, stop_list = stft(
        data,
        segment_length,
        segment_length_padded,
        shift_length,
        window_function
    )

    reconstructed = istft(
        X_stft,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    X_inv = X_stft[::-1,:]

    inv_audio = istft(
        X_inv,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    X_inv_dec = X_inv[::-1,:]

    inv_dec = istft(
        X_inv_dec,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    write_wav(os.path.join(OUT_ENC_INVERT_DIR,f"test_sound{i}_invert_enc.wav"),fs,inv_audio)
    write_wav(os.path.join(OUT_DEC_INVERT_DIR,f"test_sound{i}_invert_dec.wav"),fs,inv_dec)

    perm = np.random.permutation(X_stft.shape[0])

    X_shuffle = X_stft[perm,:]

    shuffle_audio = istft(
        X_shuffle,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    inv_perm = np.argsort(perm)

    X_shuffle_dec = X_shuffle[inv_perm,:]

    shuffle_dec = istft(
        X_shuffle_dec,
        segment_length,
        segment_length_padded,
        start_list,
        stop_list,
        original_size,
        window_function,
        p
    )

    write_wav(os.path.join(OUT_ENC_SHUFFLE_DIR,f"test_sound{i}_shuffle_enc.wav"),fs,shuffle_audio)
    write_wav(os.path.join(OUT_DEC_SHUFFLE_DIR,f"test_sound{i}_shuffle_dec.wav"),fs,shuffle_dec)